In [61]:
import pandas as pd
from os.path import join, abspath

artifacts_abspath = abspath("../../../artifacts")
cache_abspath = join(artifacts_abspath, "evm/opcodes_old.csv")
export_abspath = join(artifacts_abspath, "evm/anki_export.csv")

In [3]:
raw = pd.read_csv(cache_abspath)
raw

,uint8,Mnemonic,Stack Input 1,Stack Input 2,Stack Input 3,Stack Input 4,Stack Input 5,Stack Input 6,Stack Input 7,Stack Input 8,Stack Output 1,Stack Output 2,Stack Output 3,Stack Output 4,Stack Output 5,Stack Output 6,Expression,Notes
0,00,STOP,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,NaN,NaN,NaN,NaN,NaN,STOP(),halts execution of the contract
1,01,ADD,a,b,NaN,NaN,NaN,NaN,NaN,NaN,a + b,NaN,NaN,NaN,NaN,NaN,a + b,(u)int256 addition modulo 2**256
2,02,MUL,a,b,NaN,NaN,NaN,NaN,NaN,NaN,a * b,NaN,NaN,NaN,NaN,NaN,a * b,(u)int256 multiplication modulo 2**256
3,03,SUB,a,b,NaN,NaN,NaN,NaN,NaN,NaN,a - b,NaN,NaN,NaN,NaN,NaN,a - b,(u)int256 subtraction modulo 2**256
4,04,DIV,a,b,NaN,NaN,NaN,NaN,NaN,NaN,a // b,NaN,NaN,NaN,NaN,NaN,a // b,uint256 division
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251,FB,Invalid,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,NaN,NaN,NaN,NaN,NaN,-,-
252,FC,Invalid,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,NaN,NaN,NaN,NaN,NaN,-,-
253,FD,REVERT,offset,length,NaN,NaN,NaN,NaN,NaN,NaN,-,NaN,NaN,NaN,NaN,NaN,revert(memory[offset:offset+length]),"Byzantium hardfork, EIP-140: reverts with retu..."
254,FE,Invalid,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,NaN,NaN,NaN,NaN,NaN,-,-


In [88]:
with_inputs = raw.copy()
with_inputs.insert(
    with_inputs.columns.get_loc("Stack Input 1"),
    # with_inputs.columns.get_loc("Mnemonic"),
    "Stack Inputs",
    raw.filter(regex="^Stack Input").apply(
        lambda r: ",\n".join(r.dropna().values.astype(str)), axis=1
    ),
)
with_inputs = with_inputs.drop(
    columns=with_inputs.filter(regex=r"^Stack Input \d").columns
)
with_outputs = with_inputs.copy()
with_outputs.insert(
    with_outputs.columns.get_loc("Stack Output 1"),
    "Stack Output",
    with_outputs.filter(regex="Stack Output").apply(
        lambda r: ",\n".join(r.dropna().values.astype(str)), axis=1
    ),
)
with_outputs = with_outputs.drop(
    columns=with_outputs.filter(regex=r"Stack Output \d").columns
)
valid = with_outputs[with_outputs["Mnemonic"] != "Invalid"]
valid = valid.rename(columns={"uint8": "Opcode"})
valid

,Opcode,Mnemonic,Stack Inputs,Stack Output,Expression,Notes
0,00,STOP,-,-,STOP(),halts execution of the contract
1,01,ADD,"a,\nb",a + b,a + b,(u)int256 addition modulo 2**256
2,02,MUL,"a,\nb",a * b,a * b,(u)int256 multiplication modulo 2**256
3,03,SUB,"a,\nb",a - b,a - b,(u)int256 subtraction modulo 2**256
4,04,DIV,"a,\nb",a // b,a // b,uint256 division
...,...,...,...,...,...,...
244,F4,DELEGATECALL,"gas,\naddr,\nargsOffset,\nargsLength,\nretOffs...",success,"success, memory[retOffset:retOffset+retLength]...","Homestead hardfork, EIP-7: calls a method in a..."
245,F5,CREATE2,"value,\noffset,\nlength,\nsalt",addr,addr = new memory[offset:offset+length].value(...,"Constantinople harfork, EIP-1014: creates a ch..."
250,FA,STATICCALL,"gas,\naddr,\nargsOffset,\nargsLength,\nretOffs...",success,"success, memory[retOffset:retOffset+retLength]...","Byzantium hardfork, EIP-214: calls a method in..."
253,FD,REVERT,"offset,\nlength",-,revert(memory[offset:offset+length]),"Byzantium hardfork, EIP-140: reverts with retu..."


In [89]:
def create_back(r):
    return "\n".join(
        [
            f"<h2>{r['Mnemonic']} - {r['Opcode']}</h2>",
            "",
            f"<p>{r['Notes']}</p>",
            "",
            '<dl style="width=100%;">',
            f"  <dt>Expression</dt>",
            f"  <dd>{r['Expression']}</dd>",
            f"  <dt>Stack Inputs</dt>",
            f"  <dd>{r['Stack Inputs']}</dd>",
            f"  <dt>Stack Output</dt>",
            f"  <dd>{r['Stack Output']}</dd>",
            "</dl>",
        ]
    )

In [110]:
mnemonic = valid.copy()
mnemonic["Back"] = mnemonic.apply(create_back, axis=1)
mnemonic.rename(columns={"Mnemonic": "Front"}, inplace=True)
mnemonic["Front"] = mnemonic["Front"].apply(
    lambda v: f"<p>Mnemonic:</p>\n<p>{str(v)}</p>"
)
drop_columns = [c for c in mnemonic.columns if c not in ["Front", "Back"]]
mnemonic.drop(columns=drop_columns, inplace=True)

In [111]:
expression = valid.copy()
expression["Back"] = expression.apply(create_back, axis=1)
expression.rename(columns={"Expression": "Front"}, inplace=True)
expression["Front"] = expression["Front"].apply(
    lambda v: f"<p>Expression:</p>\n<p>{str(v)}</p>"
)
drop_columns = [c for c in expression.columns if c not in ["Front", "Back"]]
expression.drop(columns=drop_columns, inplace=True)
expression.drop_duplicates(subset=["Front"], inplace=True)
expression

,Front,Back
0,<p>Expression:</p>\n<p>STOP()</p>,<h2>STOP - 00</h2>\n\n<p>halts execution of th...
1,<p>Expression:</p>\n<p>a + b</p>,<h2>ADD - 01</h2>\n\n<p>(u)int256 addition mod...
2,<p>Expression:</p>\n<p>a * b</p>,<h2>MUL - 02</h2>\n\n<p>(u)int256 multiplicati...
3,<p>Expression:</p>\n<p>a - b</p>,<h2>SUB - 03</h2>\n\n<p>(u)int256 subtraction ...
4,<p>Expression:</p>\n<p>a // b</p>,<h2>DIV - 04</h2>\n\n<p>uint256 division</p>\n...
...,...,...
243,<p>Expression:</p>\n<p>return memory[offset:of...,<h2>RETURN - F3</h2>\n\n<p>returns from this c...
244,"<p>Expression:</p>\n<p>success, memory[retOffs...",<h2>DELEGATECALL - F4</h2>\n\n<p>Homestead har...
250,"<p>Expression:</p>\n<p>success, memory[retOffs...",<h2>STATICCALL - FA</h2>\n\n<p>Byzantium hardf...
253,<p>Expression:</p>\n<p>revert(memory[offset:of...,"<h2>REVERT - FD</h2>\n\n<p>Byzantium hardfork,..."


In [113]:
opcode = valid.copy()
opcode["Back"] = opcode.apply(create_back, axis=1)
opcode.rename(columns={"Opcode": "Front"}, inplace=True)
opcode["Front"] = opcode["Front"].apply(
    lambda v: f"<p>Opcode:</p>\n<p>{str(v)}</p>"
)
drop_columns = [c for c in opcode.columns if c not in ["Front", "Back"]]
opcode.drop(columns=drop_columns, inplace=True)

In [114]:
combined = pd.concat([mnemonic, expression, opcode], ignore_index=True)
combined.to_csv(
    export_abspath, sep=":", index=False, header=False, encoding="UTF-8"
)